# 3.1 Trajectory and Window Search

This notebook shows how partial I/Q points form a trajectory and how the readout window is selected. This is the software explanation for the hardware Trajectory Analyzer.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

DATA_PATHS = [
    Path('./s21_data.mat'),
    Path('../s21_data.mat'),
    Path('../../software/s21_data.mat'),
]

def find_s21_data():
    for p in DATA_PATHS:
        if p.exists():
            return p
    raise FileNotFoundError('Put s21_data.mat in the notebook directory or artery/software/.')

def load_s21():
    import scipy.io as sio
    data_path = find_s21_data()
    read_data = sio.loadmat(data_path)
    read_zero = read_data['data'][0]
    read_one = read_data['data'][1]
    read_zero_i, read_zero_q = read_zero[:, :, 0], read_zero[:, :, 1]
    read_one_i, read_one_q = read_one[:, :, 0], read_one[:, :, 1]
    return read_data, read_zero_i, read_zero_q, read_one_i, read_one_q

def demod_part(omega, read_i, read_q, phase=0.0):
    assert read_i.shape == read_q.shape
    ts = np.arange(read_i.shape[1])
    cos_ = np.cos(omega * ts + phase)[None, :]
    sin_ = np.sin(omega * ts + phase)[None, :]
    sum_i = np.sum(read_i * cos_ + read_q * sin_, axis=1)
    sum_q = np.sum(read_q * cos_ - read_i * sin_, axis=1)
    return np.column_stack([sum_i, sum_q])

OMEGAS = 2 * np.pi * (np.array([6.881, 6.79525, 6.97284]) - 7)

In [ ]:
read_data, read_zero_i, read_zero_q, read_one_i, read_one_q = load_s21()
window_base, window_cnt, window_len = 850, 6, 300
shots, base_shot = 2, 122

def trajectory(read_i, read_q, shot, omega):
    points = []
    for step in range(window_cnt):
        end = window_base + (step + 1) * window_len
        feat = demod_part(omega, read_i[shot:shot + 1, window_base:end], read_q[shot:shot + 1, window_base:end])
        points.append(feat[0])
    return np.array(points)

plt.figure(figsize=(6, 5))
for shot in range(shots):
    z = trajectory(read_zero_i, read_zero_q, base_shot + shot, OMEGAS[0])
    o = trajectory(read_one_i, read_one_q, base_shot + shot, OMEGAS[0])
    plt.plot(z[:, 0], z[:, 1], 'o-.', color='skyblue', label='|0>' if shot == 0 else None)
    plt.plot(o[:, 0], o[:, 1], 'o-.', color='orange', label='|1>' if shot == 0 else None)
plt.xlabel('integrated I')
plt.ylabel('integrated Q')
plt.legend()
plt.tight_layout()

## Original Notebook Figure: Demodulation Trajectory

![Original Notebook: Demodulation Trajectory](../results/3_1_demodulation_trajectory.png)

## Original Notebook Figure: Trajectory Comparison

![Original Notebook: Trajectory Comparison](../results/3_1_trajectory_comparison.png)

In [ ]:
from sklearn import metrics

train_idx = slice(0, 1000)
test_idx = slice(1000, 2000)
search_records = []
for omega_id, omega in enumerate(OMEGAS):
    for start in range(0, 1200, 100):
        for length in range(100, 2200, 100):
            if start + length > read_zero_i.shape[1]:
                continue
            train_zero = demod_part(omega, read_zero_i[train_idx, start:start + length], read_zero_q[train_idx, start:start + length])
            train_one = demod_part(omega, read_one_i[train_idx, start:start + length], read_one_q[train_idx, start:start + length])
            center_zero, center_one = train_zero.mean(axis=0), train_one.mean(axis=0)
            test_zero = demod_part(omega, read_zero_i[test_idx, start:start + length], read_zero_q[test_idx, start:start + length])
            test_one = demod_part(omega, read_one_i[test_idx, start:start + length], read_one_q[test_idx, start:start + length])
            test_data = np.vstack([test_zero, test_one])
            test_label = np.array([0] * len(test_zero) + [1] * len(test_one))
            pred = (np.linalg.norm(test_data - center_one, axis=1) < np.linalg.norm(test_data - center_zero, axis=1)).astype(int)
            search_records.append((omega_id, start, length, float(np.mean(pred == test_label))))

best = sorted(search_records, key=lambda x: x[3], reverse=True)[:10]
for item in best:
    print('omega_id=%d start=%d length=%d acc=%.4f' % item)

## ARTERY Connection

Window search decides where the streaming analyzer should start accumulating and how long it should keep evidence before the final fallback decision. A lower latency design chooses the earliest window that reaches acceptable confidence; a higher accuracy mode waits for more samples.